# 🥇 Gold Layer: Business Intelligence

### 🎯 Objective
Query the aggregated business metrics and demonstrate **Zero-Copy Isolation**.

### 🌿 Branch Strategy: `gold` vs `main`
- **Current Status**: We are currently building the Gold Layer on the `gold` branch.
- **Isolation Magic**: 
  - Users probing `main` **CANNOT** see these tables yet.
  - We can query `daily_sales_gold@gold` to validate results safely.
- **Merging**: Once we approve this notebook, we merge `gold` -> `main`.

In [ ]:
import os
from pyspark.sql import SparkSession

# Note: Connecting to the 'gold' branch by default
spark = SparkSession.builder \
    .appName("Gold_Demo") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,software.amazon.awssdk:bundle:2.17.178,software.amazon.awssdk:url-connection-client:2.17.178") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "gold") \
    .config("spark.sql.catalog.nessie.authentication.type", "NONE") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://lakehouse/warehouse") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("✅ Spark Session Connected to Gold Branch (Staging)")

### 📈 Dashboard Preview
These are the queries that will power the Executive Dashboard.

In [ ]:
# 1. Daily Sales Trend
print("📊 DAILY SALES TREND")
spark.sql("""
    SELECT order_date, total_revenue, total_orders 
    FROM nessie.ecommerce.`daily_sales_gold@gold` 
    ORDER BY order_date DESC 
    LIMIT 5
""").show()

In [ ]:
# 2. Top Brands
print("🏆 TOP BRANDS")
spark.sql("""
    SELECT brand, total_revenue 
    FROM nessie.ecommerce.`brand_performance_gold@gold` 
    ORDER BY total_revenue DESC 
    LIMIT 5
""").show()

In [ ]:
# 3. Customer Segments
print("👥 CUSTOMER SEGMENTS")
spark.sql("""
    SELECT customer_segment, count(*) as count, sum(total_spend) as revenue
    FROM nessie.ecommerce.`customer_stats_gold@gold` 
    GROUP BY customer_segment
""").show()